# Comparer le module python `hockey_scraper` avec l'API officiel

In [1]:
import contextlib
import time
import unicodedata
import warnings

import hockey_scraper as hs
import numpy as np
import pandas as pd
import requests

In [2]:
warnings.filterwarnings("ignore", category=SyntaxWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATE_FROM = "2024-10-09"
DATE_TO = "2024-10-11"
SEASON = DATE_FROM[:4] if int(DATE_FROM[5:7]) >= 9 else str(int(DATE_FROM[:4]) - 1)
GAME_TYPE = 2  # 2 = saison régulière, 3 = séries

API_STATS = "https://api.nhle.com/stats/rest/en"

## 1. Endpoint officiel

In [3]:
def fetch_stats(report, date_from, date_to, game_type=2, page=100):
    # Pagination sur l'API stats (start / limit)
    url = f"{API_STATS}/skater/{report}"
    cayenne = (
        f'gameDate<="{date_to} 23:59:59" and gameDate>="{date_from}" and gameTypeId={game_type}'
    )
    rows, start = [], 0
    while True:
        params = {
            "isAggregate": "false",
            "isGame": "true",
            "start": start,
            "limit": page,
            "sort": '[{"property":"playerId","direction":"ASC"}]',
            "cayenneExp": cayenne,
        }
        j = requests.get(url, params=params, timeout=30).json()
        rows += j["data"]
        start += page
        if start >= j["total"]:
            return pd.DataFrame(rows)


t0 = time.perf_counter()
api = fetch_stats("goalsForAgainst", DATE_FROM, DATE_TO, GAME_TYPE)
t_api = time.perf_counter() - t0
n_api = api["gameId"].nunique()
print(f"API stats : {t_api:.1f} s au total | {n_api} matchs | {t_api / n_api:.2f} s par match")
api = api.rename(columns={"gameId": "game_id"}).set_index(["game_id", "playerId"]).sort_index()
# L'API renvoie null (et non 0) quand le joueur n'était sur la glace pour aucun but
num = api.select_dtypes("number").columns.drop("evenStrengthGoalsForPct")
api[num] = api[num].fillna(0)
print(api.shape)
api.head()

API stats : 3.8 s au total | 18 matchs | 0.21 s par match
(648, 22)


assists  evenStrengthGoalDifference  evenStrengthGoalsAgainst  evenStrengthGoalsFor  evenStrengthGoalsForPct  evenStrengthTimeOnIcePerGame    gameDate  gamesPlayed  goals homeRoad       lastName  \
game_id    playerId                                                                                                                                                                                                       
2024020006 8471817         0                         0.0                       0.0                   0.0                      NaN                         446.0  2024-10-09            1      0        R         Reaves   
           8474157         0                         0.0                       0.0                   0.0                      NaN                         595.0  2024-10-09            1      0        R     Pacioretty   
           8475166         0                         0.0                       0.0                   0.0                      NaN                         868.0  2024-10-09            1      0        R        Tavares   
           8475171         0                         0.0                       0.0                   0.0                      NaN                         983.0  2024-10-09            1      0        R  Ekman-Larsson   
           8475233         0                         0.0                       0.0                   0.0                      NaN                         958.0  2024-10-09            1      0        H         Savard   

                    opponentTeamAbbrev  points positionCode  powerPlayGoalFor  powerPlayGoalsAgainst  powerPlayTimeOnIcePerGame  shortHandedGoalsAgainst  shortHandedGoalsFor  shortHandedTimeOnIcePerGame  \
game_id    playerId                                                                                                                                                                                          
2024020006 8471817                 MTL       0            R               0.0                    0.0                       10.0                      0.0                  0.0                          0.0   
           8474157                 MTL       0            L               0.0                    0.0                      209.0                      0.0                  0.0                          0.0   
           8475166                 MTL       0            C               0.0                    0.0                      260.0                      0.0                  0.0                          0.0   
           8475171                 MTL       0            D               0.0                    1.0                      201.0                      0.0                  0.0                         61.0   
           8475233                 TOR       0            D               0.0                    0.0                        0.0                      0.0                  0.0                        247.0   

                           skaterFullName teamAbbrev  
game_id    playerId                                   
2024020006 8471817            Ryan Reaves        TOR  
           8474157         Max Pacioretty        TOR  
           8475166           John Tavares        TOR  
           8475171   Oliver Ekman-Larsson        TOR  
           8475233           David Savard        MTL

## 2. Scraping `hockey_scraper`

On complète les ID manquants (noms HTML ≠ JSON, ex. `CHRIS TANEV`) à partir des shifts, qui ont toujours un `Player_Id`.
Les noms sont normalisés (encodage réparé, accents retirés, majuscules) : les shifts contiennent par ex. `JURAJ SLAFKOVSKÃ½`.
Les ID de gardiens (`Home_Goalie_Id`, `Away_Goalie_Id`) peuvent aussi être vides : sans correction, le gardien serait
compté comme patineur.

In [4]:
t0 = time.perf_counter()
res = hs.scrape_date_range(DATE_FROM, DATE_TO, if_scrape_shifts=True, data_format="pandas")
t_hs = time.perf_counter() - t0

pbp, shifts = res["pbp"].copy(), res["shifts"].copy()
for df in (pbp, shifts):
    df["game_id"] = (SEASON + df["Game_Id"].astype(str).str.zfill(6)).astype(int)

n_hs = pbp["game_id"].nunique()
print(f"hockey_scraper : {t_hs:.1f} s au total | {n_hs} matchs | {t_hs / n_hs:.2f} s par match")


def norm(name):
    if not isinstance(name, str):
        return name
    # mojibake dans hockey_scraper : UTF-8 lu comme latin-1 ("SLAFKOVSKÃ½")
    with contextlib.suppress(UnicodeEncodeError, UnicodeDecodeError):
        name = name.encode("latin-1").decode("utf-8")
    return unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode().upper().strip()


name2id = {
    (g, t, norm(n)): pid
    for g, t, n, pid in shifts[["game_id", "Team", "Player", "Player_Id"]].itertuples(index=False)
}


def fill_ids(name_col, id_col, team_col):
    miss = pbp[id_col].isna() & pbp[name_col].notna()
    keys = zip(
        pbp.loc[miss, "game_id"],
        pbp.loc[miss, team_col],
        pbp.loc[miss, name_col].map(norm),
        strict=True,
    )
    pbp.loc[miss, id_col] = [name2id.get(k, np.nan) for k in keys]
    return miss.sum()


n_filled = 0
for side, team_col in (("home", "Home_Team"), ("away", "Away_Team")):
    for i in range(1, 7):
        n_filled += fill_ids(f"{side}Player{i}", f"{side}Player{i}_id", team_col)
for i in (1, 2, 3):
    n_filled += fill_ids(f"p{i}_name", f"p{i}_ID", "Ev_Team")
n_filled += fill_ids("Home_Goalie", "Home_Goalie_Id", "Home_Team")
n_filled += fill_ids("Away_Goalie", "Away_Goalie_Id", "Away_Team")
print("ID complétés :", n_filled)

still_missing = [
    (c, pbp.loc[pbp[c].isna() & pbp[n].notna(), n].unique())
    for c, n in [
        (f"{s}Player{i}_id", f"{s}Player{i}") for s in ("home", "away") for i in range(1, 7)
    ]
]
print("Encore sans ID :", {n for _, names in still_missing for n in names} or "aucun")

goalies = set(pbp["Home_Goalie_Id"].dropna()) | set(pbp["Away_Goalie_Id"].dropna())

Scraping the schedule between 2024-10-09 and 2024-10-11...please give it a moment


Scraping Schedule: 100%|██████████| 1/1 [00:01<00:00,  1.33s/it]


Scraping Game  2024020006 2024-10-09
Scraping Game  2024020007 2024-10-09


Scraping Game  2024020008 2024-10-09
Scraping Game  2024020009 2024-10-09
Scraping Game  2024020010 2024-10-09
Scraping Game  2024020011 2024-10-10


Scraping Game  2024020012 2024-10-10
Scraping Game  2024020013 2024-10-10
Scraping Game  2024020015 2024-10-10
Scraping Game  2024020016 2024-10-10


Scraping Game  2024020014 2024-10-10


Scraping Game  2024020017 2024-10-10
Scraping Game  2024020018 2024-10-10


Scraping Game  2024020019 2024-10-10
Scraping Game  2024020020 2024-10-11
Scraping Game  2024020021 2024-10-11
Scraping Game  2024020022 2024-10-11
Scraping Game  2024020023 2024-10-11

Players missing IDs:
  - JURAJ SLAFKOVSKY 2024020006
  - CHRIS TANEV 2024020006
  - NICHOLAS ROBERTSON 2024020006
  - J.T. MILLER 2024020009
  - AATU RATY 2024020009
  - JURAJ SLAFKOVSKY 2024020011
  - CHRIS TANEV 2024020015
  - NICHOLAS ROBERTSON 2024020015
  - JUUSO VALIMAKI 2024020016
  - JOE VELENO 2024020014
  - MATHEW DUMBA 2024020017
  - EVGENII DADONOV 2024020017
  - ZACHARY ASTON-REESE 2024020018
  - YEGOR CHINAKHOV 2024020018
  - ZACK BOLDUC 2024020019
  - CAMERON ATKINSON 2024020020
  - NICHOLAS PAUL 2024020020
  - PATRICK MAROON 2024020021
  - J.T. MILLER 2024020022
  - AATU RATY 2024020022
  - ZACK BOLDUC 2024020023
  - SAM MONTEMBEAULT 20006
  - nan 20006

hockey_scraper : 142.4 s au total | 18 matchs | 7.91 s par match
ID complétés : 2664
Encore sans ID : {'NICHOLAS PAUL', 'JOE VELENO', '

## 3. Buts pour / contre par situation

`Strength` (`local x visiteur`) ne compte pas l'attaquant supplémentaire quand un gardien est retiré, ce qui correspond
à la définition NHL (un but à 6 c. 5 sans pénalité est « à forces égales »). Fusillade (période 5) exclue.

In [5]:
goals = pbp[(pbp["Event"] == "GOAL") & (pbp["Period"] < 5)].copy()
sk = goals["Strength"].str.split("x", expand=True).astype(int)
is_home = goals["Ev_Team"] == goals["Home_Team"]
for_sk = np.where(is_home, sk[0], sk[1])
ag_sk = np.where(is_home, sk[1], sk[0])
goals["goal_type"] = np.select([for_sk > ag_sk, for_sk < ag_sk], ["PP", "SH"], "EV")
print(goals["goal_type"].value_counts())

rows = []
for _, g in goals.iterrows():
    for side, team in (("home", g["Home_Team"]), ("away", g["Away_Team"])):
        for i in range(1, 7):
            pid = g[f"{side}Player{i}_id"]
            if pd.notna(pid) and pid not in goalies:
                rows.append(
                    {
                        "game_id": g["game_id"],
                        "playerId": int(pid),
                        "goal_type": g["goal_type"],
                        "fa": "For" if team == g["Ev_Team"] else "Against",
                    }
                )

on_ice = pd.DataFrame(rows)
COLS = {  # (type de but, pour/contre) -> champ API
    ("EV", "For"): "evenStrengthGoalsFor",
    ("EV", "Against"): "evenStrengthGoalsAgainst",
    ("PP", "For"): "powerPlayGoalFor",
    ("PP", "Against"): "powerPlayGoalsAgainst",
    ("SH", "For"): "shortHandedGoalsFor",
    ("SH", "Against"): "shortHandedGoalsAgainst",
}
gfa = (
    on_ice.groupby(["game_id", "playerId", "goal_type", "fa"])
    .size()
    .unstack(["goal_type", "fa"], fill_value=0)
)
gfa.columns = [COLS[c] for c in gfa.columns]
gfa = gfa.reindex(columns=list(COLS.values()), fill_value=0)

# Buts / passes
pts = pd.concat(
    {
        "goals": goals.groupby(["game_id", "p1_ID"]).size(),
        "assists": pd.concat([goals.groupby(["game_id", c]).size() for c in ("p2_ID", "p3_ID")])
        .groupby(level=[0, 1])
        .sum(),
    },
    axis=1,
).fillna(0)
pts.index = pts.index.set_levels(pts.index.levels[1].astype(int), level=1)
pts.index.names = ["game_id", "playerId"]
pts["points"] = pts["goals"] + pts["assists"]
gfa.head()

goal_type
EV    85
PP    28
SH     5
Name: count, dtype: int64


evenStrengthGoalsFor  evenStrengthGoalsAgainst  powerPlayGoalFor  powerPlayGoalsAgainst  shortHandedGoalsFor  shortHandedGoalsAgainst
game_id    playerId                                                                                                                                       
2024020006 8475171                      0                         0                 0                      1                    0                        0
           8476875                      0                         0                 1                      0                    0                        0
           8478483                      0                         0                 0                      1                    0                        0
           8479318                      0                         0                 0                      1                    0                        0
           8480018                      0                         0                 1                      0                    0                        0

## 4. Temps de glace par situation (shifts)

Pour chaque seconde de chaque période : nombre de patineurs par équipe (gardiens exclus), moins 1 si le gardien de
l'équipe est hors de la glace (attaquant supplémentaire). Égal → EV, plus → PP, moins → SH.

In [6]:
home_of = pbp.drop_duplicates("game_id").set_index("game_id")["Home_Team"]
toi_rows = []
for (gid, _per), s in shifts.groupby(["game_id", "Period"]):
    T = int(s["End"].max())
    side_of = np.where(s["Team"] == home_of[gid], "H", "A")
    skaters = {k: np.zeros(T, int) for k in "HA"}
    goalie_in = {k: np.zeros(T, bool) for k in "HA"}
    for side, (_, r) in zip(side_of, s.iterrows(), strict=True):
        a, b = int(r["Start"]), int(r["End"])
        if r["Player_Id"] in goalies:
            goalie_in[side][a:b] = True
        else:
            skaters[side][a:b] += 1
    eff = {k: skaters[k] - (~goalie_in[k]).astype(int) for k in "HA"}
    for side, (_, r) in zip(side_of, s.iterrows(), strict=True):
        if r["Player_Id"] in goalies:
            continue
        a, b = int(r["Start"]), int(r["End"])
        me, opp = eff[side][a:b], eff["A" if side == "H" else "H"][a:b]
        toi_rows.append(
            {
                "game_id": gid,
                "playerId": r["Player_Id"],
                "evenStrengthTimeOnIcePerGame": (me == opp).sum(),
                "powerPlayTimeOnIcePerGame": (me > opp).sum(),
                "shortHandedTimeOnIcePerGame": (me < opp).sum(),
            }
        )

toi = pd.DataFrame(toi_rows).groupby(["game_id", "playerId"]).sum()
toi.head()

evenStrengthTimeOnIcePerGame  powerPlayTimeOnIcePerGame  shortHandedTimeOnIcePerGame
game_id    playerId                                                                                      
2024020006 8471817                            446                         10                            0
           8474157                            595                        209                            0
           8475166                            868                        260                            0
           8475171                            983                        201                           61
           8475233                            958                          0                          247

## 5. Comparaison

In [7]:
recon = gfa.join(pts, how="outer").join(toi, how="outer").fillna(0)
recon = recon[~recon.index.get_level_values("playerId").isin(goalies)]
FIELDS = list(COLS.values()) + ["goals", "assists", "points"] + list(toi.columns)

a, r = api[FIELDS].align(recon[FIELDS], join="outer", fill_value=0)
print("Joueurs-matchs : API", len(api), "| reconstruits", len(recon), "| union", len(a))

summary = pd.DataFrame(
    {
        "somme_api": a.sum(),
        "somme_hs": r.sum(),
        "ecart_abs_moyen": (a - r).abs().mean(),
        "ecart_abs_max": (a - r).abs().max(),
    }
).round(3)
summary

Joueurs-matchs : API 648 | reconstruits 648 | union 648


,somme_api,somme_hs,ecart_abs_moyen,ecart_abs_max
evenStrengthGoalsFor,421.0,421.0,0.000,0.0
evenStrengthGoalsAgainst,422.0,421.0,0.002,1.0
powerPlayGoalFor,140.0,138.0,0.003,2.0
powerPlayGoalsAgainst,111.0,111.0,0.000,0.0
shortHandedGoalsFor,20.0,20.0,0.000,0.0
shortHandedGoalsAgainst,25.0,25.0,0.000,0.0
goals,118.0,118.0,0.000,0.0
assists,206.0,205.0,0.002,1.0
points,324.0,323.0,0.002,1.0
evenStrengthTimeOnIcePerGame,531254.0,528874.0,3.673,179.0
